# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [1]:
from pathlib import Path
import polars as pl
import duckdb
import os

DATA_DIR = Path("../data/raw")
FILE_PATH = DATA_DIR / "flood_dataset.parquet"

print("File ditemukan:", FILE_PATH.exists())
print("Ukuran file (MB):", round(FILE_PATH.stat().st_size / (1024**2), 2))
print("Polars:", pl.__version__)
print("DuckDB:", duckdb.__version__)

File ditemukan: True
Ukuran file (MB): 16.27
Polars: 2.1.0
DuckDB: 2.0.0


In [2]:
# Lazy loading menggunakan Polars
lf = pl.scan_parquet(FILE_PATH)

# Informasi kolom dan tipe data
schema = lf.collect_schema()

print("DAFTAR KOLOM DATASET")
print("=" * 50)

for column, dtype in schema.items():
    print(f"{column:<35} {dtype}")

DAFTAR KOLOM DATASET
district_id                         String
date                                Datetime(time_unit='us', time_zone=None)
year                                Int32
rain_1d                             Float64
rain_3d                             Float64
rain_7d                             Float64
rain_30d                            Float64
disc_now                            Float64
disc_7d_mean                        Float64
disc_change_3d                      Float64
month                               Int32
coastal                             Int64
flood_label                         Float64


In [3]:
sample = lf.head(10).collect()

print("10 BARIS PERTAMA")
display(sample)

10 BARIS PERTAMA


district_id,date,year,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
str,datetime[μs],i32,f64,f64,f64,f64,f64,f64,f64,i32,i64,f64
"""IDN.10.13.10_1""",2015-01-30 00:00:00,2015,13.3,35.0,62.0,348.3,25.98,26.238571,0.56,1,1,0.0
"""IDN.10.13.10_1""",2015-01-31 00:00:00,2015,12.5,44.0,67.2,348.1,26.35,25.92,2.35,1,1,0.0
"""IDN.10.13.10_1""",2015-02-01 00:00:00,2015,10.4,36.2,67.0,352.5,25.98,25.548571,1.8,2,1,0.0
"""IDN.10.13.10_1""",2015-02-02 00:00:00,2015,1.9,24.8,65.9,326.8,24.76,25.238571,-1.22,2,1,0.0
"""IDN.10.13.10_1""",2015-02-03 00:00:00,2015,20.7,33.0,80.5,327.8,25.98,25.318571,-0.37,2,1,0.0
"""IDN.10.13.10_1""",2015-02-04 00:00:00,2015,18.1,40.7,95.1,323.1,29.69,26.131429,3.71,2,1,0.0
"""IDN.10.13.10_1""",2015-02-05 00:00:00,2015,18.6,57.4,95.5,339.8,30.19,26.99,5.43,2,1,0.0
"""IDN.10.13.10_1""",2015-02-06 00:00:00,2015,38.1,74.8,120.3,356.2,36.33,28.468571,10.35,2,1,0.0
"""IDN.10.13.10_1""",2015-02-07 00:00:00,2015,27.1,83.8,134.9,380.2,37.99,30.131429,8.3,2,1,0.0


In [4]:
total_rows = lf.select(pl.len()).collect().item()
total_columns = len(schema)

print("Jumlah baris:", f"{total_rows:,}")
print("Jumlah kolom:", total_columns)

Jumlah baris: 1,152,432
Jumlah kolom: 13


In [5]:
null_counts = lf.select(
    pl.all().null_count()
).collect()

display(null_counts)

district_id,date,year,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
# Statistik dasar untuk kolom numerik
numeric_columns = [
    name
    for name, dtype in schema.items()
    if dtype.is_numeric()
]

if numeric_columns:
    statistics = lf.select(
        [
            pl.col(column).min().alias(f"{column}_min")
            for column in numeric_columns
        ] +
        [
            pl.col(column).max().alias(f"{column}_max")
            for column in numeric_columns
        ] +
        [
            pl.col(column).mean().alias(f"{column}_mean")
            for column in numeric_columns
        ]
    ).collect()

    display(statistics)
else:
    print("Tidak ditemukan kolom numerik.")

year_min,rain_1d_min,rain_3d_min,rain_7d_min,rain_30d_min,disc_now_min,disc_7d_mean_min,disc_change_3d_min,month_min,coastal_min,flood_label_min,year_max,rain_1d_max,rain_3d_max,rain_7d_max,rain_30d_max,disc_now_max,disc_7d_mean_max,disc_change_3d_max,month_max,coastal_max,flood_label_max,year_mean,rain_1d_mean,rain_3d_mean,rain_7d_mean,rain_30d_mean,disc_now_mean,disc_7d_mean_mean,disc_change_3d_mean,month_mean,coastal_mean,flood_label_mean
i32,f64,f64,f64,f64,f64,f64,f64,i32,i64,f64,i32,f64,f64,f64,f64,f64,f64,f64,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2015,0.0,0.0,0.0,0.0,0.0,0.0,-758.73,1,0,0.0,2024,258.7,415.7,645.2,1502.5,1467.93,1307.318571,1076.38,12,1,1.0,2019.536424,6.793898,20.382072,47.561081,203.643537,20.809407,20.806564,0.000984,6.566501,0.899371,0.073159


In [7]:
duplicate_count = (
    lf.select(pl.struct(pl.all()).is_duplicated().sum())
    .collect()
    .item()
)

print("Jumlah baris duplikat:", f"{duplicate_count:,}")

Jumlah baris duplikat: 0


In [8]:
con = duckdb.connect()

con.execute(f"""
    CREATE VIEW flood_data AS
    SELECT *
    FROM read_parquet('{FILE_PATH.as_posix()}')
""")

summary = con.execute("""
    SUMMARIZE flood_data
""").df()

display(summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,district_id,VARCHAR,IDN.10.13.10_1,IDN.9.9.9_1,316,NaN,NaN,NaN,NaN,NaN,1152432,0.0
1,date,TIMESTAMP,2015-01-30 00:00:00,2024-12-31 00:00:00,3712,2020-01-15 12:00:00,NaN,2017-07-23 03:35:52.340014,2020-01-15 22:21:26.672156,2022-07-09 02:44:46.356104,1152432,0.0
2,year,INTEGER,2015,2024,10,2019.5364238410596,2.8556717940747394,2017,2020,2022,1152432,0.0
3,rain_1d,DOUBLE,0.0,258.7,1332,6.793897687671181,10.10181064199661,0.3102058265651896,2.9782538095282907,9.777347223340776,1152432,0.0
4,rain_3d,DOUBLE,0.0,415.70000000000005,54437,20.382071566912654,23.51372439659673,2.201293751937434,13.672714821062801,30.881690418216024,1152432,0.0
5,rain_7d,DOUBLE,0.0,645.2,43777,47.561081174425716,46.715256295523545,8.41853931595645,37.61793087214816,72.18495608160401,1152432,0.0
6,rain_30d,DOUBLE,0.0,1502.5,30705,203.64353714578917,165.6060230160802,60.07664049156212,179.39753267769353,310.20342533246475,1152432,0.0
7,disc_now,DOUBLE,0.0,1467.93,2863,20.809406672146086,71.05971098343126,0.7225371787735888,2.7145676263902923,10.669058074500233,1152432,0.0
8,disc_7d_mean,DOUBLE,0.0,1307.3185714285714,183910,20.806564402186886,70.15783131833687,0.7626605869124681,2.7895883919333695,10.820594251990162,1152432,0.0
9,disc_change_3d,DOUBLE,-758.7300000000001,1076.38,53862,0.0009839279020340383,18.861786504809054,-0.6191748743394475,-0.03021803170823922,0.3096824373378738,1152432,0.0


In [9]:
total_rows = lf.select(pl.len()).collect().item()
total_columns = len(schema)

print(f"Total baris   : {total_rows:,}")
print(f"Total kolom   : {total_columns}")
print(f"Ukuran file   : {FILE_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"Memenuhi syarat README: {total_rows > 1_000_000}")

Total baris   : 1,152,432
Total kolom   : 13
Ukuran file   : 16.27 MB
Memenuhi syarat README: True


In [10]:
display(lf.head(10).collect())

district_id,date,year,rain_1d,rain_3d,rain_7d,rain_30d,disc_now,disc_7d_mean,disc_change_3d,month,coastal,flood_label
str,datetime[μs],i32,f64,f64,f64,f64,f64,f64,f64,i32,i64,f64
"""IDN.10.13.10_1""",2015-01-30 00:00:00,2015,13.3,35.0,62.0,348.3,25.98,26.238571,0.56,1,1,0.0
"""IDN.10.13.10_1""",2015-01-31 00:00:00,2015,12.5,44.0,67.2,348.1,26.35,25.92,2.35,1,1,0.0
"""IDN.10.13.10_1""",2015-02-01 00:00:00,2015,10.4,36.2,67.0,352.5,25.98,25.548571,1.8,2,1,0.0
"""IDN.10.13.10_1""",2015-02-02 00:00:00,2015,1.9,24.8,65.9,326.8,24.76,25.238571,-1.22,2,1,0.0
"""IDN.10.13.10_1""",2015-02-03 00:00:00,2015,20.7,33.0,80.5,327.8,25.98,25.318571,-0.37,2,1,0.0
"""IDN.10.13.10_1""",2015-02-04 00:00:00,2015,18.1,40.7,95.1,323.1,29.69,26.131429,3.71,2,1,0.0
"""IDN.10.13.10_1""",2015-02-05 00:00:00,2015,18.6,57.4,95.5,339.8,30.19,26.99,5.43,2,1,0.0
"""IDN.10.13.10_1""",2015-02-06 00:00:00,2015,38.1,74.8,120.3,356.2,36.33,28.468571,10.35,2,1,0.0
"""IDN.10.13.10_1""",2015-02-07 00:00:00,2015,27.1,83.8,134.9,380.2,37.99,30.131429,8.3,2,1,0.0


In [11]:
duplicate_count = (
    lf.select(
        pl.struct(pl.all()).is_duplicated().sum()
    )
    .collect()
    .item()
)

print(f"Jumlah baris duplikat: {duplicate_count:,}")

Jumlah baris duplikat: 0


In [12]:
label_distribution = (
    lf.group_by("flood_label")
    .agg(pl.len().alias("jumlah_baris"))
    .sort("flood_label")
    .with_columns(
        (
            pl.col("jumlah_baris") /
            pl.col("jumlah_baris").sum() * 100
        ).round(2).alias("persentase")
    )
    .collect()
)

display(label_distribution)

flood_label,jumlah_baris,persentase
f64,u32,f64
0.0,1068121,92.68
1.0,84311,7.32


In [13]:
yearly_data = (
    lf.group_by("year")
    .agg(pl.len().alias("jumlah_baris"))
    .sort("year")
    .collect()
)

display(yearly_data)

year,jumlah_baris
i32,u32
2015,106848
2016,116388
2017,116070
2018,116070
2019,116070
2020,116388
2021,116070
2022,116070
2023,116070


In [14]:
district_data = (
    lf.group_by("district_id")
    .agg(pl.len().alias("jumlah_baris"))
    .sort("jumlah_baris", descending=True)
    .collect()
)

display(district_data.head(10))

district_id,jumlah_baris
str,u32
"""IDN.9.23.6_1""",3624
"""IDN.9.23.7_1""",3624
"""IDN.9.23.8_1""",3624
"""IDN.9.23.9_1""",3624
"""IDN.9.4.10_1""",3624
"""IDN.9.4.11_1""",3624
"""IDN.9.4.12_1""",3624
"""IDN.9.4.13_1""",3624
"""IDN.9.4.14_1""",3624


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [ ]:
lazy_data = pl.scan_parquet(DATA_PATH)
print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

In [ ]:
contoh_data = lazy_data.head(10).collect()
contoh_data

## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:
connection = duckdb.connect()
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_parquet(?)', [str(DATA_PATH)]
).pl()
profil_duckdb

## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.

## Temuan Awal Dataset

### 1. Ukuran dan Jumlah Data
- Nama file: `flood_dataset.parquet`
- Ukuran file: sekitar 16,27 MB.
- Jumlah baris: 1.152.432 baris.
- Jumlah kolom: 13 kolom.
- Periode data: 2015–2024.
- Cakupan wilayah: Pantai Utara Jawa (Pantura), Indonesia.
- Jumlah distrik: 316 distrik.

Dataset memenuhi persyaratan tugas karena memiliki lebih dari 1 juta baris.

### 2. Tipe Kolom

Dataset memiliki tipe kolom sebagai berikut:

| Kolom | Tipe Data |
|---|---|
| `district_id` | String |
| `date` | Datetime |
| `year` | Integer |
| `month` | Integer |
| `rain_1d` | Float |
| `rain_3d` | Float |
| `rain_7d` | Float |
| `rain_30d` | Float |
| `disc_now` | Float |
| `disc_7d_mean` | Float |
| `disc_change_3d` | Float |
| `coastal` | Integer |
| `flood_label` | Float |

### 3. Kolom dengan Missing Values Terbanyak

Hasil pemeriksaan menunjukkan bahwa seluruh 13 kolom memiliki 0 missing values atau nilai kosong.

Dengan demikian, tidak ditemukan kolom dengan missing values terbanyak karena seluruh kolom memiliki jumlah null yang sama, yaitu nol.

### 4. Anomali yang Perlu Diperiksa pada Milestone 2

Beberapa kondisi yang perlu diperiksa lebih lanjut dalam proses cleaning dan validasi data:

1. **Potensi outlier:** Beberapa variabel curah hujan dan indikator lainnya memiliki nilai ekstrem berdasarkan metode Interquartile Range (IQR).
2. **Konsistensi curah hujan antarperiode:** Ditemukan beberapa observasi dengan nilai `rain_3d` lebih kecil daripada `rain_1d`, `rain_7d` lebih kecil daripada `rain_3d`, atau `rain_30d` lebih kecil daripada `rain_7d`. Kondisi ini perlu diperiksa berdasarkan definisi variabel.
3. **Validitas nilai numerik:** Perlu dipastikan tidak terdapat nilai negatif yang tidak masuk akal pada variabel curah hujan dan indikator yang relevan.
4. **Duplikasi data:** Perlu diperiksa kemungkinan baris duplikat dan duplikasi kombinasi `district_id` dengan `date`.
5. **Konsistensi waktu:** Perlu divalidasi kesesuaian antara kolom `date`, `year`, dan `month`.
6. **Validitas label:** Kolom `flood_label` dan `coastal` perlu dipastikan hanya memiliki nilai kategori biner 0 dan 1.

### 5. Kesimpulan Awal

Secara umum, dataset memiliki ukuran yang memenuhi persyaratan tugas dan struktur kolom yang dapat digunakan untuk analisis temporal serta perbandingan antarwilayah. Tidak ditemukan missing values pada pemeriksaan awal. Namun, potensi outlier dan konsistensi antarvariabel tetap perlu diperiksa lebih lanjut pada Milestone 2 sebelum data digunakan untuk analisis lanjutan.